# Step 2: Preprocessing

Tujuan: Membersihkan teks dari noise agar lebih suitable untuk feature extraction.

**Flow untuk Text Categorization (Sentiment Analysis):**
```
Lowercase -> Hapus links/username -> Hapus tanda baca -> Hapus whitespace berlebih
```

**Catatan penting:**
- Stop words **TIDAK** dihapus — penting untuk sentiment analysis (kata seperti "not", "no", "very" mempengaruhi sentimen)
- Stemming/Lemmatization **TIDAK** dilakukan di tahap ini (bisa dicoba di iterasi berikutnya)

In [1]:
import pandas as pd
import string
import re
import warnings
warnings.filterwarnings('ignore')

## 1. Load Data dari EDA

In [2]:
df = pd.read_csv('../data/raw/Tweets.csv')

# Pastikan kolom yang dibutuhkan ada
assert 'text' in df.columns, 'Kolom text tidak ditemukan'
assert 'airline_sentiment' in df.columns, 'Kolom airline_sentiment tidak ditemukan'

print(f'Jumlah data: {len(df)}')
print(f'\nContoh data sebelum preprocessing:')
df[['text', 'airline_sentiment']].head()

Jumlah data: 14640

Contoh data sebelum preprocessing:


,text,airline_sentiment
0,@VirginAmerica What @dhepburn said.,neutral
1,@VirginAmerica plus you've added commercials t...,positive
2,@VirginAmerica I didn't today... Must mean I n...,neutral
3,@VirginAmerica it's really aggressive to blast...,negative
4,@VirginAmerica and it's a really big bad thing...,negative


**Hasil:**

Dataset berhasil di-load. Kita hanya akan menggunakan 2 kolom:
- `text` — teks tweet mentah (fitur)
- `airline_sentiment` — label sentimen (target)

Terlihat semua tweet diawali dengan `@VirginAmerica` (mention) — ini akan dihapus saat preprocessing.

## 2. Definisi Fungsi Preprocessing

In [3]:
def preprocess_text(text):
    """
    Fungsi preprocessing untuk sentiment analysis.
    
    Flow: lowercase -> hapus links -> hapus username -> hapus hashtag ->
          hapus tanda baca -> hapus angka -> hapus whitespace berlebih
    
    Catatan: Stop words TIDAK dihapus (penting untuk sentiment).
    """
    # 1. Lowercase — menyamakan case agar "Flight" dan "flight" dianggap sama
    text = text.lower()
    
    # 2. Hapus links (http, https, www) — URL tidak informatif untuk sentiment
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    
    # 3. Hapus username (@) — semua tweet diawali @, tidak ada informasi sentimen
    text = re.sub(r'@\w+', '', text)
    
    # 4. Hapus hashtag (#) — simpan kata, hapus simbol #
    text = re.sub(r'#\w+', '', text)
    
    # 5. Hapus tanda baca — simbol seperti . , ! ? tidak informatif
    text = text.translate(str.maketrans('', '', string.punctuation))
    
    # 6. Hapus angka — angka jarang mempengaruhi sentimen
    text = re.sub(r'\d+', '', text)
    
    # 7. Hapus whitespace berlebih — spasi ganda setelah penghapusan
    text = ' '.join(text.split())
    
    return text

**Penjelasan setiap langkah preprocessing:**

| Langkah | Mengapa | Contoh |
|---------|---------|--------|
| Lowercase | Menyamakan case agar kata sama tidak dianggap berbeda | Flight → flight |
| Hapus links | URL tidak mengandung informasi sentimen | https://t.co/abc → (kosong) |
| Hapus username | Semua tweet diawali @airline, tidak informatif | @united → (kosong) |
| Hapus hashtag | Simbol # dihapus, tapi kata tetap tersimpan | #delayed → delayed |
| Hapus tanda baca | Simbol seperti . , ! ? tidak relevan | great! → great |
| Hapus angka | Angka jarang mempengaruhi sentimen | 30 → (kosong) |
| Hapus whitespace | Spasi ganda setelah penghapusan | hello  world → hello world |

**Yang TIDAK dilakukan:**
- **Stop words removal** — kata seperti "not", "no", "very" penting untuk sentiment
- **Stemming/Lemmatization** — bisa dicoba di iterasi berikutnya

## 3. Terapkan Preprocessing

In [4]:
df['clean_text'] = df['text'].apply(preprocess_text)

print('Preprocessing selesai!')
print(f'\nContoh hasil preprocessing:')
comparison = df[['text', 'clean_text', 'airline_sentiment']].head(5)
for idx, row in comparison.iterrows():
    print(f'\n--- Tweet {idx} ({row["airline_sentiment"]}) ---')
    print(f'Before: {row["text"]}')
    print(f'After:  {row["clean_text"]}')

Preprocessing selesai!

Contoh hasil preprocessing:

--- Tweet 0 (neutral) ---
Before: @VirginAmerica What @dhepburn said.
After:  what said

--- Tweet 1 (positive) ---
Before: @VirginAmerica plus you've added commercials to the experience... tacky.
After:  plus youve added commercials to the experience tacky

--- Tweet 2 (neutral) ---
Before: @VirginAmerica I didn't today... Must mean I need to take another trip!
After:  i didnt today must mean i need to take another trip

--- Tweet 3 (negative) ---
Before: @VirginAmerica it's really aggressive to blast obnoxious "entertainment" in your guests' faces &amp; they have little recourse
After:  its really aggressive to blast obnoxious entertainment in your guests faces amp they have little recourse

--- Tweet 4 (negative) ---
Before: @VirginAmerica and it's a really big bad thing about it
After:  and its a really big bad thing about it


**Hasil:**

Perbandingan sebelum dan sesudah preprocessing pada 5 tweet pertama:

| Tweet | Sebelum | Sesudah |
|-------|---------|----------|
| 0 (neutral) | @VirginAmerica What @dhepburn said. | what said |
| 1 (positive) | @VirginAmerica plus you've added commercials to the experience... tacky. | plus youve added commercials to the experience tacky |
| 2 (neutral) | @VirginAmerica I didn't today... Must mean I need to take another trip! | i didnt today must mean i need to take another trip |
| 3 (negative) | @VirginAmerica it's really aggressive to blast obnoxious "entertainment" in your guests' faces &amp; they have little recourse | its really aggressive to blast obnoxious entertainment in your guests faces amp they have little recourse |
| 4 (negative) | @VirginAmerica and it's a really big bad thing about it | and its a really big bad thing about it |

**Yang berubah:**
- `@VirginAmerica` dihapus dari semua tweet
- Tanda baca dihapus: `.`, `'`, `!`, `"`, `&amp;`
- Huruf kapital → lowercase: `What` → `what`, `Must` → `must`
- Kontraksi dipotong: `you've` → `youve`, `it's` → `its`, `didn't` → `didnt`

## 4. Cek Hasil Preprocessing

In [5]:
# Cek panjang teks sebelum vs sesudah
df['original_length'] = df['text'].apply(len)
df['clean_length'] = df['clean_text'].apply(len)

print('=== Perbandingan Panjang Teks ===')
print(f'Rata-rata panjang sebelum: {df["original_length"].mean():.1f} karakter')
print(f'Rata-rata panjang sesudah:  {df["clean_length"].mean():.1f} karakter')
print(f'Reduksi rata-rata: {(1 - df["clean_length"].mean()/df["original_length"].mean())*100:.1f}%')

# Cek apakah ada teks kosong setelah preprocessing
empty_count = (df['clean_text'].str.strip() == '').sum()
print(f'\nTeks kosong setelah preprocessing: {empty_count}')

=== Perbandingan Panjang Teks ===
Rata-rata panjang sebelum: 103.8 karakter
Rata-rata panjang sesudah:  81.8 karakter
Reduksi rata-rata: 21.2%

Teks kosong setelah preprocessing: 0


**Hasil:**

| Metrik | Sebelum | Sesudah | Perubahan |
|--------|---------|---------|----------|
| Rata-rata panjang (karakter) | 103.8 | 81.8 | -21.2% |
| Teks kosong | - | 0 | Tidak ada |

- **Reduksi 21.2%** — wajar karena @mention, links, tanda baca, dan angka dihapus
- **Teks kosong = 0** — semua tweet masih memiliki konten setelah dibersihkan
- Tidak ada tweet yang isinya hanya @mention atau links saja

In [6]:
# Cek sample per sentimen setelah preprocessing
for sent in ['positive', 'neutral', 'negative']:
    print(f'\n=== Sample {sent.upper()} setelah preprocessing ===')
    subset = df[df['airline_sentiment'] == sent]
    for i, text in enumerate(subset['clean_text'].head(2).values):
        print(f'{i+1}. {text}')


=== Sample POSITIVE setelah preprocessing ===
1. plus youve added commercials to the experience tacky
2. yes nearly every time i fly vx this “ear worm” won’t go away

=== Sample NEUTRAL setelah preprocessing ===
1. what said
2. i didnt today must mean i need to take another trip

=== Sample NEGATIVE setelah preprocessing ===
1. its really aggressive to blast obnoxious entertainment in your guests faces amp they have little recourse
2. and its a really big bad thing about it


**Hasil:**

Setelah preprocessing, teks menjadi lebih bersih dan seragam:

| Sentimen | Contoh Hasil Preprocessing |
|----------|---------------------------|
| Positive | `plus youve added commercials to the experience tacky` |
| Neutral | `what said` |
| Negative | `its really aggressive to blast obnoxious entertainment in your guests faces amp they have little recourse` |

**Observasi:**
- Teks neutral cenderung **lebih pendek** (hanya 2 kata: "what said")
- Teks negative cenderung **lebih panjang** dan mengandung kata-kata keluhan
- Kata "amp" muncul karena `&amp;` (HTML entity) dihapus simbol &-nya, menyisakan `amp`

Teks sekarang siap untuk feature extraction (TF-IDF).

## 5. Simpan Hasil Preprocessing

In [7]:
# Simpan hanya kolom yang diperlukan
df_clean = df[['airline_sentiment', 'clean_text']].copy()
df_clean.to_csv('../data/processed/cleaned_data.csv', index=False)

print(f'Data tersimpan di: data/processed/cleaned_data.csv')
print(f'Jumlah data: {len(df_clean)}')
print(f'Kolom: {df_clean.columns.tolist()}')
print(f'\nDistribusi label setelah preprocessing:')
print(df_clean['airline_sentiment'].value_counts())

Data tersimpan di: data/processed/cleaned_data.csv
Jumlah data: 14640
Kolom: ['airline_sentiment', 'clean_text']

Distribusi label setelah preprocessing:
airline_sentiment
negative    9178
neutral     3099
positive    2363
Name: count, dtype: int64


**Hasil:**

File `cleaned_data.csv` berhasil disimpan:

| Info | Nilai |
|------|-------|
| Jumlah baris | 14.640 |
| Kolom | `airline_sentiment`, `clean_text` |
| Negative | 9.178 (62.69%) |
| Neutral | 3.099 (21.17%) |
| Positive | 2.363 (16.14%) |

- Jumlah data **tetap 14.640** — tidak ada yang dihapus
- Distribusi label **tetap sama** seperti sebelum preprocessing
- File ini akan digunakan sebagai input untuk notebook berikutnya (`03_feature_extraction.ipynb`)

## Ringkasan Preprocessing

### Yang Dilakukan:
1. **Lowercase** — menyamakan semua huruf ke kecil
2. **Hapus links** — URL (http/https/www) dihapus
3. **Hapus username** — @mention dihapus
4. **Hapus hashtag** — simbol # dihapus, kata tetap tersimpan
5. **Hapus tanda baca** — simbol seperti . , ! ? & dihapus
6. **Hapus angka** — digit dihapus
7. **Hapus whitespace** — spasi ganda dihapus

### Hasil:
- Reduksi panjang teks rata-rata: **21.2%** (103.8 → 81.8 karakter)
- Teks kosong setelah preprocessing: **0** (semua tweet masih memiliki konten)
- Distribusi label tetap: negative 9.178, neutral 3.099, positive 2.363

### Yang TIDAK Dilakukan:
- **Stop words removal** — kata seperti "not", "no", "very" penting untuk sentiment
- **Stemming/Lemmatization** — bisa dicoba di iterasi berikutnya

### File Output:
- `data/processed/cleaned_data.csv` — data bersih siap untuk feature extraction

**Lanjut ke notebook berikutnya: `03_feature_extraction.ipynb`**